# 04 - Evaluation

Reproduces every number reported in the paper, from the released corpus and the
released model weights. Nothing here retrains anything, so it runs on CPU in a
few minutes.

Three evaluations:

1. **Lexicon baseline** on the held-out silver test set
2. **BiLSTM-CRF** on the same test set
3. **BiLSTM-CRF on external gold data** - Ghosh et al. (2025), which the model
   has never seen

Run from `notebooks/`. All paths are relative to the repository root.

In [ ]:
import sys, os, json, collections
from pathlib import Path

ROOT = Path.cwd().parent          # repository root
sys.path.insert(0, str(ROOT / "src"))

from model import (load_tagger, tag_sentence, read_conll, read_conllu,
                   score, print_report)

CORPUS   = ROOT / "data" / "corpus"
EXTERNAL = ROOT / "data" / "external" / "iiit"
MODELDIR = ROOT / "models" / "pos_tagger"
RESULTS  = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

for p in [CORPUS, EXTERNAL, MODELDIR]:
    print("%-10s %s  %s" % (p.name, "OK " if p.exists() else "MISSING", p))

## 1. Lexicon baseline

Most-frequent-tag lookup built from the training split only. Types absent from
the lexicon fall back to the corpus-wide most frequent tag.

In [ ]:
test = read_conll(CORPUS / "test.conll")
print("test sentences: %d" % len(test))

# The full training split is not redistributed. The released lexicon holds the
# most frequent training tag for every word type, which is all the baseline uses.
# If you have train.conll, the lexicon is rebuilt from it instead.
train_path = CORPUS / "train.conll"
if train_path.exists():
    counts, totals = collections.defaultdict(collections.Counter), collections.Counter()
    for toks, tags in read_conll(train_path):
        for t, g in zip(toks, tags):
            counts[t.lower()][g] += 1
            totals[g] += 1
    lexicon = {w: c.most_common(1)[0][0] for w, c in counts.items()}
    default = totals.most_common(1)[0][0]
    print("lexicon built from train.conll")
else:
    lexicon = {}
    with open(ROOT / "data" / "lexicon" / "baseline_lexicon.tsv", encoding="utf-8") as f:
        for line in f:
            w, g = line.rstrip("\n").split("\t")
            lexicon[w] = g
    default = json.loads((ROOT / "data" / "stats" / "train_tag_counts.json")
                         .read_text(encoding="utf-8"))["default_tag"]
    print("lexicon loaded from data/lexicon/baseline_lexicon.tsv")
print("lexicon entries: %d    OOV default: %s" % (len(lexicon), default))

pairs, oov = [], 0
for toks, tags in test:
    for t, g in zip(toks, tags):
        p = lexicon.get(t.lower())
        if p is None:
            p, oov = default, oov + 1
        pairs.append((g, p))

lex_res = score(pairs)
lex_res["oov_tokens"] = oov
lex_res["lexicon_size"] = len(lexicon)
print_report(lex_res, "LEXICON BASELINE (silver test set)")
print("\nOOV: %d of %d (%.1f%%)" % (oov, lex_res["tokens"],
                                    100 * oov / lex_res["tokens"]))


## 2. BiLSTM-CRF on the silver test set

In [ ]:
model, vocabs, device = load_tagger(MODELDIR)
print("device:", device)

pairs = []
for toks, tags in test:
    pred = tag_sentence(model, vocabs, device, toks)
    pairs += list(zip(tags, pred[:len(tags)]))

nn_res = score(pairs)
print_report(nn_res, "BiLSTM-CRF (silver test set)")

## 3. Comparison

The macro averages invert because of `NUM`, which occurs twice in the test set.
Both figures are reported.

In [ ]:
def macro_excl(res, drop):
    v = [m["f1"] for t, m in res["per_tag"].items() if t != drop]
    return sum(v) / len(v)

print("%-28s %10s %12s" % ("", "Lexicon", "BiLSTM-CRF"))
print("-" * 52)
print("%-28s %9.2f%% %11.2f%%" % ("Token accuracy",
      100 * lex_res["accuracy"], 100 * nn_res["accuracy"]))
print("%-28s %10.3f %12.3f" % ("Macro F1",
      lex_res["macro_f1"], nn_res["macro_f1"]))
print("%-28s %10.3f %12.3f" % ("Macro F1 excl. NUM",
      macro_excl(lex_res, "NUM"), macro_excl(nn_res, "NUM")))
print("%-28s %10.3f %12.3f" % ("Weighted F1",
      lex_res["weighted_f1"], nn_res["weighted_f1"]))

print("\nPer-tag F1")
print("  %-8s %9s %11s %9s" % ("Tag", "Lexicon", "BiLSTM-CRF", "Delta"))
print("  " + "-" * 42)
for t in sorted(nn_res["per_tag"]):
    a = lex_res["per_tag"].get(t, {}).get("f1", 0.0)
    b = nn_res["per_tag"][t]["f1"]
    print("  %-8s %9.4f %11.4f %+9.4f" % (t, a, b, b - a))

## 4. External gold evaluation

Ghosh et al. (2025), 502 human-annotated sentences. The model was trained on none
of them, so all three splits are used.

Punctuation was stripped from our training corpus, so the punctuation-excluded
figure is the meaningful one. Both are reported.

In [ ]:
# The gold data of Ghosh et al. (2025) is not redistributed here.
# Obtain it from its authors and place train.conllu, val.conllu and
# test.conllu in data/external/iiit/ to run this section.
gold = []
for name in ["train.conllu", "val.conllu", "test.conllu"]:
    p = EXTERNAL / name
    if p.exists():
        s = read_conllu(p)
        gold += s
        print("  %-14s %4d sentences" % (name, len(s)))
    else:
        print("  %-14s missing" % name)

EXT_OK = len(gold) > 0
if not EXT_OK:
    print("\nExternal gold data not found; sections 4 and 6 are skipped.")
else:
    print("\ntotal: %d sentences, %d tokens"
          % (len(gold), sum(len(t) for t, _ in gold)))

    pairs = []
    for toks, tags in gold:
        pred = tag_sentence(model, vocabs, device, toks)
        pairs += list(zip(tags, pred[:len(tags)]))

    ext_all   = score(pairs)
    ext_nopun = score([(g, p) for g, p in pairs if g != "PUNCT"])

    print_report(ext_all,   "EXTERNAL GOLD - all tokens")
    print_report(ext_nopun, "EXTERNAL GOLD - punctuation excluded")

    print("\n===== TOP CONFUSIONS (gold -> predicted) =====")
    conf = collections.Counter((g, p) for g, p in pairs if g != p and g != "PUNCT")
    for (g, p), k in conf.most_common(15):
        print("  %-6s -> %-6s %5d" % (g, p, k))

## 5. Save

In [ ]:
# Merge into the existing results file so that entries from sections
# that were skipped (for example the external evaluation) are kept.
path = RESULTS / "all_evaluations.json"
out = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
out["lexicon_baseline"] = lex_res
out["bilstm_crf"] = nn_res
if EXT_OK:
    out["external_gold_all"] = ext_all
    out["external_gold_nopunct"] = ext_nopun
    out["external_confusions"] = {"%s->%s" % k: v for k, v in conf.most_common(30)}
path.write_text(json.dumps(out, indent=2), encoding="utf-8")
print("saved:", path)


## 6. Scheme-neutral evaluation

In [ ]:
if not EXT_OK:
    print('External gold data not found; section 6 skipped.')
else:
    # ============================================================
    # 6. Scheme-neutral evaluation
    #
    # Append this as a NEW CELL at the end of 04_evaluation.ipynb and run it.
    # It reuses `pairs` (gold, predicted) built in section 4, plus `RESULTS`.
    #
    # Purpose: separate genuine tagging errors from annotation-scheme
    # disagreement. Two collapses are applied, each motivated by a documented
    # difference between our annotation and that of Ghosh et al. (2025).
    # ============================================================

    import json
    import collections

    # ---- the two documented divergences --------------------------------------
    #
    # PRED: Mizo has no morphologically distinct adjective class. Property
    #       concepts are stative verbs, and most adverbs derive from them.
    #       Our scheme favours ADJ; theirs distributes across VERB and ADV.
    #
    # FUNC: Mizo marks auxiliation, determination and adposition-like relations
    #       with particles. Our scheme routes these to PART; theirs keeps the
    #       separate UD categories.
    #
    COLLAPSE = {
        "ADJ": "PRED", "VERB": "PRED", "ADV": "PRED",
        "PART": "FUNC", "AUX": "FUNC", "DET": "FUNC",
        "ADP": "FUNC", "SCONJ": "FUNC",
    }

    # PRON is deliberately NOT merged into FUNC. The particle *a* is tagged PART
    # by us and PRON by them (545 tokens). That is a substantive analytical
    # disagreement, not a granularity difference, so it stays counted as an error.

    MALFORMED = {"_"}


    def remap(tag):
        return COLLAPSE.get(tag, tag)


    base = [(g, p) for g, p in pairs if g != "PUNCT" and g not in MALFORMED]
    mapped = [(remap(g), remap(p)) for g, p in base]

    raw_res = score(base)
    map_res = score(mapped)

    print_report(raw_res, "ORIGINAL TAGSET (punctuation excluded)")
    print_report(map_res, "SCHEME-NEUTRAL TAGSET")

    print("\n%-34s %8s %8s" % ("", "Original", "Mapped"))
    print("-" * 52)
    print("%-34s %7.2f%% %7.2f%%" % ("Token accuracy",
          100 * raw_res["accuracy"], 100 * map_res["accuracy"]))
    print("%-34s %8.3f %8.3f" % ("Macro F1", raw_res["macro_f1"], map_res["macro_f1"]))
    print("%-34s %8.3f %8.3f" % ("Weighted F1",
          raw_res["weighted_f1"], map_res["weighted_f1"]))

    gain = 100 * (map_res["accuracy"] - raw_res["accuracy"])
    print("\nAccuracy recovered by collapsing scheme differences: %+.2f pp" % gain)


    # ---- how much each collapse contributes on its own ------------------------
    print("\n===== CONTRIBUTION OF EACH COLLAPSE =====")
    GROUPS = {
        "PRED (ADJ/VERB/ADV)":            {"ADJ", "VERB", "ADV"},
        "FUNC (PART/AUX/DET/ADP/SCONJ)":  {"PART", "AUX", "DET", "ADP", "SCONJ"},
    }
    for name, members in GROUPS.items():
        m = {t: name for t in members}
        only = [(m.get(g, g), m.get(p, p)) for g, p in base]
        r = score(only)
        print("  %-32s %6.2f%%  (%+.2f pp)"
              % (name, 100 * r["accuracy"],
                 100 * (r["accuracy"] - raw_res["accuracy"])))


    # ---- what is still wrong after mapping ------------------------------------
    print("\n===== RESIDUAL CONFUSIONS AFTER MAPPING =====")
    print("These are disagreements that scheme granularity does not explain.")
    resid = collections.Counter((g, p) for g, p in mapped if g != p)
    tot_err = sum(resid.values())
    for (g, p), k in resid.most_common(12):
        print("  %-6s -> %-6s %5d   %5.1f%% of remaining errors"
              % (g, p, k, 100 * k / tot_err))


    # ---- the PRON/PART disagreement, isolated ---------------------------------
    pron_part = sum(1 for g, p in base if g == "PRON" and p == "PART")
    print("\nPRON tagged PART (the particle *a*): %d tokens, %.1f%% of all tokens"
          % (pron_part, 100 * pron_part / len(base)))
    print("Merging PRON into FUNC as well would give %.2f%% accuracy."
          % (100 * score([({"PRON": "FUNC"}.get(g, remap(g)),
                           {"PRON": "FUNC"}.get(p, remap(p)))
                          for g, p in base])["accuracy"]))


    # ---- save ------------------------------------------------------------------
    path = RESULTS / "all_evaluations.json"
    out = json.loads(path.read_text(encoding="utf-8"))
    out["external_scheme_neutral"] = map_res
    out["external_raw_nopunct_clean"] = raw_res
    out["external_residual_confusions"] = {
        "%s->%s" % k: v for k, v in resid.most_common(20)
    }
    path.write_text(json.dumps(out, indent=2), encoding="utf-8")
    print("\nUpdated:", path)